In [4]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import requests
import pandas as pd
import time
from Bio import Entrez
from src.ena_fetcher import search_ena_studies, fetch_runs_for_study, resolve_host_species, fetch_study_origin, fetch_pubmed_abstract
from src.fetcher import configure_entrez

configure_entrez()

Entrez configured with email: akharya@ucsd.edu


In [5]:
# search for all animal WGS studies
query = 'tax_tree(33208) AND library_strategy="WGS"'
study_accessions = search_ena_studies(query, max_results=500)
print(f"Found {len(study_accessions)} unique studies")

Found 5000 runs, 547 unique studies
Found 500 unique studies


In [6]:
results = []

for study in study_accessions:
    print(f"Fetching {study}...")
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue 

    results.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().iloc[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().iloc[0] if len(runs_df["host_body_site"].dropna()) > 0 else None,
        "country": runs_df["country"].dropna().iloc[0] if len(runs_df["country"].dropna()) > 0 else None,
        "platform": runs_df["instrument_platform"].dropna().iloc[0] if "instrument_platform" in runs_df.columns and len(runs_df["instrument_platform"].dropna()) > 0 else None,
        "n_samples": runs_df["sample_accession"].nunique(),
    })

print(f"  host: {results[-1]['host_species']} | n_samples: {results[-1]['n_samples']}")

print(f"\nCollected {len(results)} studies")
pd.DataFrame(results)

Fetching PRJDB3182...
Fetching PRJDB10007...
Fetching PRJDB10555...
Fetching PRJDB12484...
Fetching PRJDB13090...
Fetching PRJDB19220...
Fetching PRJDB3439...
Fetching PRJDB3836...
Fetching PRJDB15958...
Fetching PRJDB16890...
Fetching PRJDB35936...
Fetching PRJDB5062...
Fetching PRJDB10190...
Fetching PRJDB9937...
Fetching PRJDB1831...
Fetching PRJDB2865...
Fetching PRJDB28192...
Fetching PRJDB16726...
Fetching PRJDB18684...
Fetching PRJDB35782...
Fetching PRJDB5403...
Fetching PRJDB17698...
Fetching PRJDB12457...
Fetching PRJDB13066...
Fetching PRJDB8174...
Fetching PRJDB35959...
Fetching PRJDB5055...
Fetching PRJDB13944...
Fetching PRJDB4697...
Fetching PRJDB10499...
Fetching PRJDB1558...
Fetching PRJDB16625...
Fetching PRJDB17515...
Fetching PRJDB19285...
Fetching PRJDB6951...
Fetching PRJDB9376...
Fetching PRJDB38261...
Fetching PRJDB5118...
Fetching PRJDA72881...
Fetching PRJDB5865...
Fetching PRJDB19938...
Fetching PRJDB9879...
Fetching PRJDB10538...
Fetching PRJDB18082...
Fetch

KeyboardInterrupt: 

In [ ]:
for study in study_accessions[:3]:
    print(f"Fetching {study}...")
    runs_df = fetch_runs_for_study(study)
    print(f"Got {len(runs_df) if runs_df is not None else 0} runs")

Fetching PRJDB11895...
Got 2 runs
Fetching PRJDB12484...
Got 2 runs
Fetching PRJDB2132...
Got 25 runs


In [ ]:
# get titles for all studies
for result in results:
    origin = fetch_study_origin(result['study_accession'])
    result['title'] = origin.get("title")
    print(f"{result['study_accession']}: {result['title']}")
    time.sleep(0.3)

animals_df = pd.DataFrame(results)
animals_df = animals_df[[
    "study_accession", 
    "host_species", 
    "body_site", 
    "country", 
    
    "n_samples",
    "title",
    "platform" 
]]

import os
os.makedirs("../results", exist_ok=True)
animals_df.to_csv("../results/animal_wgs_catalog.tsv", sep="\t", index=False)


print(animals_df.head(10).to_string())

PRJDB11895: Genome sequencing of the NIES strain of Chironomus yoshimatsui
PRJDB12484: Identification of gene insertion sites in transgenic mice overexpressing mouse carbonyl reductase 1
PRJDB2132: Whole genome sequencing of tammar wallaby
PRJDB2374: Whole genome of Gasterosteus wheatlandi female (Hiseq2000)
PRJDB2359: RNA Sequencing to improve annotation and detect expression changes during anhydrobiosis
PRJDB17515: Comparative genomics of the Hondo red fox
PRJDB11116: Genome sequencing of Turritopsis dohrnii
PRJDB2660: Whole genome sequencing reveals many genetic variations in the Japanese native cattle (Bos taurus) Mishima-Ushi
PRJDB7612: PacBio Sequencing of a Little Campbell Stream male
PRJDB5781: Thermobia domestica genome sequencing
PRJDB15975: Various breeds of domestic cat (Felis catus) whole genome sequence
PRJDB18423: Genome sequencing of insecticide-resistant strains of the bed bug Cimex lectularius.
PRJDB14144: Coat color dilution in Kumamoto sub-breed of Japanese Brown ca

In [ ]:
print(f"Total studies: {len(animals_df)}")
print(f"\nColumns: {list(animals_df.columns)}")
print(f"\nSample of data:")
print(animals_df.head(5).to_string())
print(f"\nMissing values:")
print(animals_df.isnull().sum())

Total studies: 500

Columns: ['study_accession', 'host_species', 'body_site', 'country', 'n_samples', 'title', 'platform']

Sample of data:
  study_accession               host_species body_site        country  n_samples                                                                                                title platform
0      PRJDB11895     Chironomus yoshimatsui            Japan:Tsukuba          1                                       Genome sequencing of the NIES strain of Chironomus yoshimatsui     None
1      PRJDB12484               Mus musculus                                   2  Identification of gene insertion sites in transgenic mice overexpressing mouse carbonyl reductase 1     None
2       PRJDB2132       Notamacropus eugenii                                   1                                                            Whole genome sequencing of tammar wallaby     None
3       PRJDB2374    Gasterosteus wheatlandi                                   1                

In [ ]:
print(animals_df["body_site"].value_counts(dropna=False))
print(f"Empty strings: {(animals_df['body_site'] == '').sum()}")

body_site
    500
Name: count, dtype: int64
Empty strings: 500


This was searched with library strategy = WGS but a bunch of these studies might not be microbiome studies at all or might just be whole genome sequences of one specific animal instead of being a sequence of a colony. The body_site column being entirely empty does not help.

In [ ]:
queries = [
    'tax_tree(33208) AND library_strategy="METAGENOMIC"',
]

all_study_accessions = set()
for query in queries:
    accessions = search_ena_studies(query, max_results=500)
    all_study_accessions.update(accessions)
    print(f"Query: {query} → {len(accessions)} studies")

study_accessions = list(all_study_accessions)
print(f"\nTotal unique studies: {len(study_accessions)}")

Query: tax_tree(33208) AND library_strategy="METAGENOMIC" → 0 studies

Total unique studies: 0


In [ ]:
# check what library_strategy values exist in ENA for animal studies
params = {
    "result": "read_run",
    "query": 'tax_tree(33208)',
    "fields": "study_accession,library_strategy",
    "limit": 100,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
data = response.json()
df = pd.DataFrame(data)
print(df["library_strategy"].value_counts())

library_strategy
FL-cDNA     56
ChIP-Seq    32
OTHER       11
EST          1
Name: count, dtype: int64


In [ ]:
params = {
    "result": "read_run",
    "query": 'tax_tree(33208)',
    "fields": "study_accession,scientific_name,library_strategy",
    "limit": 10,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
data = response.json()
df = pd.DataFrame(data)
print(df[["scientific_name", "library_strategy"]])

  scientific_name library_strategy
0    Homo sapiens          FL-cDNA
1    Homo sapiens          FL-cDNA
2    Homo sapiens          FL-cDNA
3    Homo sapiens          FL-cDNA
4    Homo sapiens          FL-cDNA
5    Homo sapiens          FL-cDNA
6    Homo sapiens          FL-cDNA
7    Homo sapiens          FL-cDNA
8    Homo sapiens          FL-cDNA
9    Homo sapiens          FL-cDNA


In [ ]:
# check library strategies in your existing results
strategies = []
for result in results[:20]:
    runs_df = fetch_runs_for_study(result["study_accession"])
    if runs_df is not None:
        strategy = runs_df["library_strategy"].value_counts().index[0]
        strategies.append(strategy)
        print(f"{result['study_accession']}: {strategy}")

PRJDB11895: WGS
PRJDB12484: WGS
PRJDB2132: WGS
PRJDB2374: WGS
PRJDB2359: WGS
PRJDB17515: WGS
PRJDB11116: RNA-Seq
PRJDB2660: WGS
PRJDB7612: WGS
PRJDB5781: WGS
PRJDB15975: WGS
PRJDB18423: WGS
PRJDB14144: WGS
PRJDB7096: WGS
PRJDB13891: WGS
PRJDB9980: WGS
PRJDB15346: WGS
PRJDB13886: WGS
PRJDB5507: WGS
PRJDB37530: WGS


In [ ]:
# try host is not empty using different syntax
params = {
    "result": "read_run",
    "query": 'tax_tree(33208) AND library_strategy="WGS"',
    "fields": "study_accession,scientific_name,host,host_scientific_name",
    "limit": 10,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
data = response.json()
df = pd.DataFrame(data)
print(df[["scientific_name", "host", "host_scientific_name"]])

Status: 200
   scientific_name host host_scientific_name
0       Bos taurus                          
1     Mus musculus                          
2     Homo sapiens                          
3     Homo sapiens                          
4  Oryzias latipes                          
5     Homo sapiens                          
6     Homo sapiens                          
7     Homo sapiens                          
8     Homo sapiens                          
9     Homo sapiens                          


In [ ]:
# filter to keep only actual metagenome studies
filtered_results = []

for result in results:
    study = result["study_accession"]
    runs_df = fetch_runs_for_study(study)
    if runs_df is None:
        continue
    
    sci_names = runs_df["scientific_name"].dropna().unique()
    host_names = runs_df["host_scientific_name"].dropna()
    host_names = host_names[host_names != ""].unique()
    
    is_metagenome = any("metagenome" in name.lower() for name in sci_names)
    has_host = len(host_names) > 0
    
    if is_metagenome or has_host:
        filtered_results.append(result)

print(f"Before filter: {len(results)}")
print(f"After filter: {len(filtered_results)}")

Before filter: 500
After filter: 15


In [ ]:
# search specifically for gut metagenome WGS studies
query = 'tax_tree(749906) AND library_strategy="WGS"'
study_accessions_wgs = search_ena_studies(query, max_results=500)
print(f"Found {len(study_accessions_wgs)} gut metagenome WGS studies")

Found 5000 runs, 66 unique studies
Found 66 gut metagenome WGS studies


In [ ]:
results_wgs = []

for study in study_accessions_wgs:
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    results_wgs.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
    })

# show host breakdown
df_wgs = pd.DataFrame(results_wgs)
print(df_wgs["host_species"].value_counts().head(20))

host_species
Homo sapiens                        5
Sus scrofa                          3
Gallus gallus                       3
Canis lupus familiaris              2
Anas platyrhynchos                  1
Mus musculus                        1
Sus scrofa domesticus               1
Pseudois nayaur                     1
Meleagris gallopavo                 1
Naja atra                           1
Phascolarctos cinereus              1
Apis cerana                         1
Lagopus muta hyperborea             1
Vibrio cholerae                     1
Papio cynocephalus and P. anubis    1
Name: count, dtype: int64


In [ ]:
unresolved = [r for r in results_wgs if r["host_species"] is None]
print(f"Unresolved studies: {len(unresolved)}")
for r in unresolved[:5]:
    runs_df = fetch_runs_for_study(r["study_accession"])
    print(f"\n{r['study_accession']}:")
    print(f"  scientific_name: {runs_df['scientific_name'].value_counts().index[0]}")
    print(f"  host: {runs_df['host'].dropna().unique()[:3]}")
    print(f"  host_scientific_name: {runs_df['host_scientific_name'].dropna().unique()[:3]}")

Unresolved studies: 42

PRJEB14847:
  scientific_name: gut metagenome
  host: ['']
  host_scientific_name: ['']

PRJEB58786:
  scientific_name: gut metagenome
  host: ['']
  host_scientific_name: ['']

PRJEB35362:
  scientific_name: gut metagenome
  host: ['']
  host_scientific_name: ['']

PRJEB47458:
  scientific_name: metagenome
  host: ['']
  host_scientific_name: ['']

PRJEB102780:
  scientific_name: gut metagenome
  host: ['']
  host_scientific_name: ['']


In [ ]:
# add titles
for result in results_wgs:
    provenance = fetch_study_origin(result["study_accession"])
    result["title"] = provenance.get("title")
    time.sleep(0.3)

# build final DataFrame
catalog_df = pd.DataFrame(results_wgs)
catalog_df = catalog_df[[
    "study_accession",
    "host_species",
    "body_site",
    "country",
    "n_samples",
    "title"
]]

# save to TSV
import os
os.makedirs("../results", exist_ok=True)
catalog_df.to_csv("../results/animal_wgs_catalog.tsv", sep="\t", index=False)

print(f"Saved {len(catalog_df)} studies")
print(catalog_df.to_string())

Saved 66 studies
   study_accession                      host_species     body_site                                       country  n_samples                                                                                                                                                                                                                                                                                            title
0       PRJEB14847                              None                                                                    323                                                                                                                                                                                                                                                         International Human Microbiome Standards
1        PRJEB6921                      Homo sapiens                                                                    709                                  

In [ ]:
print(df_wgs["host_species"].value_counts(dropna=False))
print(f"\nStudies with no host resolved: {df_wgs['host_species'].isna().sum()}")

host_species
None                                42
Homo sapiens                         5
Sus scrofa                           3
Gallus gallus                        3
Canis lupus familiaris               2
Anas platyrhynchos                   1
Mus musculus                         1
Sus scrofa domesticus                1
Pseudois nayaur                      1
Meleagris gallopavo                  1
Naja atra                            1
Phascolarctos cinereus               1
Apis cerana                          1
Lagopus muta hyperborea              1
Vibrio cholerae                      1
Papio cynocephalus and P. anubis     1
Name: count, dtype: int64

Studies with no host resolved: 42


In [ ]:
print(df_wgs[df_wgs["host_species"].notna()][["study_accession", "host_species"]].to_string())

   study_accession                      host_species
1        PRJEB6921                      Homo sapiens
2       PRJDB14208                      Homo sapiens
5       PRJEB20308            Canis lupus familiaris
6       PRJDB34988                Anas platyrhynchos
10      PRJDB36191                      Mus musculus
15      PRJEB26961             Sus scrofa domesticus
17      PRJEB86261                        Sus scrofa
22      PRJDB31830                   Pseudois nayaur
26      PRJEB86260               Meleagris gallopavo
28      PRJEB59610            Canis lupus familiaris
30      PRJDB38623                         Naja atra
34      PRJEB86259                     Gallus gallus
35      PRJDB34615            Phascolarctos cinereus
42      PRJEB86255                     Gallus gallus
44       PRJDB3161                      Homo sapiens
46      PRJEB23147                      Homo sapiens
47      PRJDB38794                       Apis cerana
48      PRJEB86258                     Gallus 

In [8]:
queries = [
    'tax_tree(749906) AND library_strategy="WGS"',     # gut metagenome WGS
    'tax_tree(410657) AND library_strategy="WGS"',     # metagenome WGS
    'tax_tree(1861841) AND library_strategy="WGS"',    # fecal metagenome
    'tax_tree(506600) AND library_strategy="WGS"' ,    # chicken gut metagenome
    'tax_tree(410661) AND library_strategy="WGS"',     # mouse gut metagenome 
    'tax_tree(1510822) AND library_strategy="WGS"',    # pig gut metagenome
    'tax_tree(1602388) AND library_strategy="WGS"',    # fish gut metagenome
    'tax_tree(408170) AND library_strategy="WGS"',     # human gut metagenome 
]

all_accessions = set()
for query in queries:
    accessions = search_ena_studies(query, max_results=500)
    all_accessions.update(accessions)
    print(f"{query.split('tax_tree(')[1].split(')')[0]} → {len(accessions)} studies")
    time.sleep(1)

print(f"\nTotal unique studies: {len(all_accessions)}")

Found 5000 runs, 66 unique studies
749906 → 66 studies
Found 5000 runs, 258 unique studies
410657 → 258 studies
Found 5000 runs, 204 unique studies
1861841 → 204 studies
Found 5000 runs, 89 unique studies
506600 → 89 studies
Found 5000 runs, 77 unique studies
410661 → 77 studies
Found 5000 runs, 112 unique studies
1510822 → 112 studies
Found 3152 runs, 62 unique studies
1602388 → 62 studies
Found 5000 runs, 70 unique studies
408170 → 70 studies

Total unique studies: 929


In [9]:
results_all = []

for i, study in enumerate(all_accessions):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(all_accessions)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    results_all.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
    })

print(f"\nCollected {len(results_all)} studies")

Progress: 0/929
Progress: 50/929
Progress: 100/929
Progress: 150/929
Progress: 200/929
Progress: 250/929
Progress: 300/929
Progress: 350/929
Progress: 400/929
Progress: 450/929
Progress: 500/929
Progress: 550/929
Progress: 600/929
Progress: 650/929
Progress: 700/929
Progress: 750/929
Progress: 800/929
Progress: 850/929
Progress: 900/929

Collected 929 studies


In [10]:
df_all = pd.DataFrame(results_all)

print(f"Total studies: {len(df_all)}")
print(f"With host resolved: {df_all['host_species'].notna().sum()}")
print(f"Without host: {df_all['host_species'].isna().sum()}")
print(f"\nTop host species:")
print(df_all['host_species'].value_counts().head(20))

Total studies: 929
With host resolved: 726
Without host: 203

Top host species:
host_species
Homo sapiens                    104
Sus scrofa                       61
Gallus gallus                    58
Mus musculus                     42
feces metagenome                 35
Sus scrofa domesticus            22
missing                          21
fish gut metagenome              16
pig gut metagenome               14
seawater metagenome              13
chicken gut metagenome           12
activated sludge metagenome      12
bioreactor metagenome            10
aquatic metagenome                8
sediment metagenome               8
bioreactor sludge metagenome      7
wastewater metagenome             7
Salmo salar                       6
biofilm metagenome                6
mouse                             6
Name: count, dtype: int64


In [11]:
# separate actual animals from generic metagenome terms
generic_terms = [
    "metagenome", "feces metagenome", "fish gut metagenome", "pig gut metagenome",
    "seawater metagenome", "chicken gut metagenome", "activated sludge metagenome",
    "bioreactor metagenome", "aquatic metagenome", "sediment metagenome",
    "bioreactor sludge metagenome", "wastewater metagenome", "biofilm metagenome",
    "missing", "mouse"
]

# flag each row
df_all["is_animal"] = df_all["host_species"].apply(
    lambda x: False if (pd.isna(x) or any(term in str(x).lower() for term in generic_terms)) else True
)

print(f"Confirmed animals: {df_all['is_animal'].sum()}")
print(f"Generic/unknown: {(~df_all['is_animal']).sum()}")
print(f"\nAll unique host values that are NOT animals:")
print(df_all[~df_all['is_animal']]["host_species"].value_counts())

Confirmed animals: 483
Generic/unknown: 446

All unique host values that are NOT animals:
host_species
feces metagenome                 35
missing                          21
fish gut metagenome              16
pig gut metagenome               14
seawater metagenome              13
activated sludge metagenome      12
chicken gut metagenome           12
bioreactor metagenome            10
sediment metagenome               8
aquatic metagenome                8
wastewater metagenome             7
bioreactor sludge metagenome      7
biofilm metagenome                6
mouse                             6
marine sediment metagenome        5
food fermentation metagenome      4
hot springs metagenome            4
hydrothermal vent metagenome      3
groundwater metagenome            3
sludge metagenome                 3
riverine metagenome               3
clinical metagenome               3
lake water metagenome             3
air metagenome                    3
anaerobic digester metagenome    

In [13]:
animal_df = df_all[df_all["is_animal"]].copy()
print(f"Total confirmed animal studies: {len(animal_df)}")
print(f"\nFull host species breakdown:")
print(animal_df["host_species"].value_counts().to_string())

Total confirmed animal studies: 483

Full host species breakdown:
host_species
Homo sapiens                                                          104
Sus scrofa                                                             61
Gallus gallus                                                          58
Mus musculus                                                           42
Sus scrofa domesticus                                                  22
Salmo salar                                                             6
Bos taurus                                                              6
rat                                                                     5
Gallus gallus domesticus                                                5
Danio rerio                                                             4
Ochotona curzoniae                                                      4
Canis lupus familiaris                                                  3
Micropterus salmoides            

In [14]:
non_animals = [
    "Escherichia coli", "Vibrio cholerae", "Escherichia coli HS",
    "Oryza sativa", "Arabidopsis thaliana", "Glycine max", 
    "Raphanus sativus", "Caragana korshinskii", "Oryza sativa L. cv. Nipponbare",
    "environment", "feces", "host333", "189", "not applicable-XM19C1",
    "Cow-dung", "Domestic animals", "child", "wildlife"
]

clean_df = animal_df[~animal_df["host_species"].isin(non_animals)].copy()
print(f"After removing non-animals: {len(clean_df)} studies")
print(f"Unique host species: {clean_df['host_species'].nunique()}")

After removing non-animals: 462 studies
Unique host species: 135


In [15]:
print("Fetching titles...")
titles = {}
for i, study in enumerate(clean_df["study_accession"]):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(clean_df)}")
    provenance = fetch_study_origin(study)
    titles[study] = provenance.get("title")
    time.sleep(0.3)

clean_df["title"] = clean_df["study_accession"].map(titles)

# output TSV
import os
os.makedirs("../results", exist_ok=True)
clean_df.to_csv("../results/animal_wgs_catalog.tsv", sep="\t", index=False)

print(f"\nSaved {len(clean_df)} studies to animal_wgs_catalog.tsv")
print(clean_df.head(5).to_string())

Fetching titles...
Progress: 0/462
Progress: 50/462
Progress: 100/462
Progress: 150/462
Progress: 200/462
Progress: 250/462
Progress: 300/462
Progress: 350/462
Progress: 400/462
Progress: 450/462

Saved 462 studies to animal_wgs_catalog.tsv
  study_accession   host_species host_tax_id       body_site                   country  n_samples library_strategy  is_animal                                                                                                                            title
1     PRJNA801645     Sus scrofa        9823                            Canada: Quebec         12              WGS       True                                                                      Study of the effects of a stabilizer on microbiome analysis
4      PRJEB47613  Gallus gallus        9031  Caecum content                     Spain         47              WGS       True                                                       HoloFood Chicken Caecum+Ileum Metagenome – non-targeted metabolomics 

In [16]:
def resolve_host_from_abstract(study_accession, client):
   
    origin = fetch_study_origin(study_accession)
    abstract = fetch_pubmed_abstract(study_accession)

    title = origin.get("title") or ""
    description = origin.get("description") or ""
    abstract_text = abstract or ""

    if not any([title, description, abstract_text]):
        return None
    
    prompt = f""" 
Based on the following study information, what animal species was studied 
for this study?
Title: {title}
Description: {description}
Abstract: {abstract_text}

Return only the scientific name of the host animal (e.g. "Mus musculus"). 
If this is not an animal study or you cannot determin the host, return "unknown". 
"""

    try:
        message = client.messages.create(
            model = "claude-opus-4-20250514", 
            max_tokens = 256, 
            system = "You are a taxonomist. Extract the host animal species from scientific text.",
            messages = [
                {
                    "role": "user", 
                    "content": prompt + "\n\n Please check: Is there a host \
                        species present in the title? Does it match the abstract\
                        ?"
                }
            ]
        )
        result = message.content[0].text.strip()
        return result 
    
    except Exception as e:
        print(f"Error calling Claude: {e}")
        return None

In [1]:
import pandas as pd
df = pd.read_csv("../results/host_enriched_462studies.csv")
print(f"Shape: {df.shape}")
df.head(20)

Shape: (462, 5)


,study_accession,derived_host_species,derivation_source,title,abstract
0,PRJNA801645,NaN,llm_unknown,Study of the effects of a stabilizer on microb...,NaN
1,PRJEB47613,Gallus gallus,regex_title_only_no_abstract,HoloFood Chicken Caecum+Ileum Metagenome – non...,NaN
2,PRJDB10675,NaN,llm_unknown,International collaborative studies on the pre...,NaN
3,PRJNA835973,Mus musculus,llm,Lactobacillus rhamnosus Probio-M9 alleviates c...,NaN
4,PRJEB86258,Gallus gallus,regex_title_only_no_abstract,3D'omics: Salmonella challenge chicken trial,NaN
5,PRJNA383926,Gallus gallus,regex_title_only_no_abstract,Gallus gallus Metagenome,NaN
6,PRJNA433811,Salmo salar,regex_title_only_no_abstract,shotgun metagenome sequencing salmo salar gut ...,NaN
7,PRJNA589661,NaN,llm_unknown,"Environmental samples' Metagenome: Water, soil...",NaN
8,PRJNA293646,Gallus gallus,llm,"Chicken, pig and cattle gut microbiome raw seq...",NaN
9,PRJDB4176,Homo sapiens,regex_title_only_no_abstract,metagenomic data from human fecal samples,NaN
